# Semana 6 – Bias/Variance, Overfitting y Regularización
**CADI Deep Learning – Actividad 6**

**Dataset:** Diabetes (Pima Indians Diabetes Database) vía `fetch_openml`  
**Objetivo:** Comparar un modelo base (sin regularización) contra un modelo regularizado (Dropout + L2 + Early Stopping), observando el trade-off bias/variance a través de las curvas de entrenamiento y las métricas de evaluación.

## 1. Importación de librerías

In [ ]:
# ── Librerías numéricas y de visualización ──────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

# ── TensorFlow / Keras para construir las redes neuronales ──────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers

# ── Scikit-learn: dataset, partición, escalado y métricas ───────────────────
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report
)

# ── Semilla para reproducibilidad ───────────────────────────────────────────
np.random.seed(42)
tf.random.set_seed(42)

## 2. Carga y exploración del dataset

Usamos el dataset **Pima Indians Diabetes** (id `"diabetes"` en OpenML).  
Contiene 768 muestras con 8 características clínicas y una etiqueta binaria: *tested_positive* / *tested_negative*.

In [ ]:
# Descarga el dataset desde OpenML (se cachea localmente tras la primera descarga)
diabetes = fetch_openml(name="diabetes", version=1, as_frame=False, parser="auto")

# Separamos features (X) y etiquetas (y)
X = diabetes.data.astype(np.float32)          # Matriz (768, 8)
y_raw = diabetes.target                        # Array de strings: 'tested_positive' / 'tested_negative'

# Convertimos etiquetas a valores binarios: 1 = positivo, 0 = negativo
y = (y_raw == 'tested_positive').astype(np.int32)

# Información básica del dataset
print("Shape X:", X.shape)
print("Shape y:", y.shape)
print("Clases y balance:", dict(zip(*np.unique(y, return_counts=True))))
print("Feature names:", diabetes.feature_names)

## 3. Preprocesamiento de los datos

### 3.1 División en conjuntos de entrenamiento y prueba  
Usamos 80 % para entrenamiento y 20 % para prueba, con estratificación para mantener el balance de clases.

In [ ]:
# División train/test con estratificación (mantiene proporción de clases)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,       # 20 % para evaluación final
    random_state=42,
    stratify=y            # Garantiza la misma proporción de clases en ambos conjuntos
)

print(f"Train: {X_train.shape[0]} muestras | Test: {X_test.shape[0]} muestras")
print(f"Balance train → 0: {(y_train==0).sum()} | 1: {(y_train==1).sum()}")
print(f"Balance test  → 0: {(y_test==0).sum()}  | 1: {(y_test==1).sum()}")

### 3.2 Estandarización de features  
Aplicamos `StandardScaler` (media 0, std 1). El scaler se **ajusta sólo sobre train** y luego se aplica a test, evitando *data leakage*.

In [ ]:
# Instanciamos el escalador
scaler = StandardScaler()

# fit_transform en train: calcula media/std del conjunto de entrenamiento y escala
X_train_s = scaler.fit_transform(X_train)

# transform en test: aplica la misma media/std aprendida (NO recalcula)
X_test_s  = scaler.transform(X_test)

print("Media de X_train_s (debe ser ~0):", X_train_s.mean(axis=0).round(3))
print("Std  de X_train_s (debe ser ~1):", X_train_s.std(axis=0).round(3))

## 4. Definición de los modelos

Ambos modelos tienen la **misma arquitectura base** (128 → 64 → 1) para que la comparación sea válida.  
La única diferencia son las técnicas de regularización aplicadas al segundo modelo.

| Técnica | Modelo Base | Modelo Regularizado |
|---|---|---|
| L2 (weight decay) | ✗ | ✓ (`λ=1e-4`) |
| Dropout | ✗ | ✓ (30 %) |
| Early Stopping | ✗ | ✓ (patience=10) |

In [ ]:
# ── Modelo BASE: alta capacidad, sin regularización ─────────────────────────
def build_base_model():
    model = keras.Sequential([
        # Capa de entrada: define la dimensión de las features (8 columnas)
        layers.Input(shape=(X_train_s.shape[1],)),

        # Capa oculta 1: 128 neuronas con activación ReLU
        layers.Dense(128, activation="relu"),

        # Capa oculta 2: 64 neuronas con activación ReLU
        layers.Dense(64, activation="relu"),

        # Capa de salida: 1 neurona con sigmoid → probabilidad de clase positiva
        layers.Dense(1, activation="sigmoid")
    ])
    # Compilamos con Adam, pérdida binaria y accuracy como métrica
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model


# ── Modelo REGULARIZADO: misma arquitectura + L2 + Dropout ──────────────────
def build_reg_model():
    model = keras.Sequential([
        # Capa de entrada
        layers.Input(shape=(X_train_s.shape[1],)),

        # Capa oculta 1: L2 penaliza pesos grandes → reduce overfitting
        layers.Dense(128, activation="relu",
                     kernel_regularizer=regularizers.l2(1e-4)),

        # Dropout 30 %: apaga aleatoriamente neuronas durante entrenamiento → generalización
        layers.Dropout(0.3),

        # Capa oculta 2: también con L2
        layers.Dense(64, activation="relu",
                     kernel_regularizer=regularizers.l2(1e-4)),

        # Segundo Dropout
        layers.Dropout(0.3),

        # Capa de salida: igual que el modelo base
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model


# Early Stopping: detiene el entrenamiento si val_loss no mejora en 10 épocas
# restore_best_weights=True → recupera los pesos del mejor epoch al terminar
early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

# Mostramos los resúmenes de ambos modelos
print("=== Modelo BASE ===")
build_base_model().summary()
print("\n=== Modelo REGULARIZADO ===")
build_reg_model().summary()

## 5. Entrenamiento

Ambos modelos se entrenan con los mismos hiperparámetros (épocas, batch size, split de validación) para asegurar una comparación justa.

In [ ]:
# ── Entrenamiento Modelo BASE ────────────────────────────────────────────────
model_base = build_base_model()

history_base = model_base.fit(
    X_train_s, y_train,
    validation_split=0.2,   # 20 % de train se usa como validación durante entrenamiento
    epochs=100,             # Máximo de épocas
    batch_size=32,          # Tamaño del mini-batch
    verbose=0               # Sin output por época para mantener el notebook limpio
)

# Evaluación final en el conjunto de TEST (datos no vistos)
loss_base, acc_base = model_base.evaluate(X_test_s, y_test, verbose=0)
print(f"[BASE]  Test loss: {loss_base:.4f} | Test accuracy: {acc_base:.4f}")
print(f"[BASE]  Mejor val_loss: {min(history_base.history['val_loss']):.4f}")
print(f"[BASE]  Épocas entrenadas: {len(history_base.history['loss'])}")

In [ ]:
# ── Entrenamiento Modelo REGULARIZADO ────────────────────────────────────────
model_reg = build_reg_model()

history_reg = model_reg.fit(
    X_train_s, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],  # Early stopping activo solo en el modelo regularizado
    verbose=0
)

loss_reg, acc_reg = model_reg.evaluate(X_test_s, y_test, verbose=0)
print(f"[REG]   Test loss: {loss_reg:.4f} | Test accuracy: {acc_reg:.4f}")
print(f"[REG]   Mejor val_loss: {min(history_reg.history['val_loss']):.4f}")
print(f"[REG]   Épocas entrenadas (early stopping): {len(history_reg.history['loss'])}")

## 6. Gráfica 1 – Curvas de aprendizaje (train vs validation loss)

Esta gráfica es la evidencia visual central del trade-off **bias/variance**:  
- Si `train_loss ≪ val_loss` → el modelo está en **overfitting** (alta varianza).  
- Si ambas curvas convergen cerca → el modelo **generaliza bien**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Subgráfica izquierda: Modelo BASE ───────────────────────────────────────
ax = axes[0]
ax.plot(history_base.history["loss"],     label="Train loss",      color="steelblue",  lw=2)
ax.plot(history_base.history["val_loss"], label="Val loss",        color="tomato",     lw=2, linestyle="--")
ax.set_title("Modelo BASE\n(sin regularización)", fontsize=13, fontweight="bold")
ax.set_xlabel("Época")
ax.set_ylabel("Binary Cross-Entropy Loss")
ax.legend()
ax.grid(alpha=0.3)

# ── Subgráfica derecha: Modelo REGULARIZADO ──────────────────────────────────
ax = axes[1]
ax.plot(history_reg.history["loss"],     label="Train loss",       color="steelblue",  lw=2)
ax.plot(history_reg.history["val_loss"], label="Val loss",         color="tomato",     lw=2, linestyle="--")
# Línea vertical en el epoch donde paró el early stopping
best_epoch = np.argmin(history_reg.history["val_loss"])
ax.axvline(best_epoch, color="green", linestyle=":", lw=1.8, label=f"Mejor epoch ({best_epoch})")
ax.set_title("Modelo REGULARIZADO\n(L2 + Dropout + Early Stopping)", fontsize=13, fontweight="bold")
ax.set_xlabel("Época")
ax.set_ylabel("Binary Cross-Entropy Loss")
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle("Gráfica 1 – Curvas de aprendizaje: Bias/Variance Trade-off",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfica guardada como learning_curves.png")

## 7. Métricas de evaluación detalladas

Para una evaluación completa en un problema de clasificación binaria (especialmente con posible desbalance) calculamos:  
Accuracy, Precision, Recall, F1-Score, Specificity, Sensitivity y AUC-ROC.

In [ ]:
def evaluate_model(model, X_test, y_test, name="Modelo"):
    """
    Calcula y muestra un reporte completo de métricas de clasificación.
    Retorna las predicciones binarias para su uso posterior.
    """
    # Predicción de probabilidades → umbral 0.5 para obtener clases binarias
    y_prob = model.predict(X_test, verbose=0).ravel()
    y_pred = (y_prob >= 0.5).astype(int)

    # Matriz de confusión para derivar TP, TN, FP, FN
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    # Métricas estándar de clasificación binaria
    acc        = accuracy_score(y_test, y_pred)
    precision  = precision_score(y_test, y_pred, zero_division=0)
    recall     = recall_score(y_test, y_pred)          # Sensitivity = TP / (TP+FN)
    f1         = f1_score(y_test, y_pred)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0  # TN / (TN+FP)
    auc        = roc_auc_score(y_test, y_prob)

    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  Accuracy    : {acc:.4f}")
    print(f"  Precision   : {precision:.4f}")
    print(f"  Recall/Sens.: {recall:.4f}")
    print(f"  Specificity : {specificity:.4f}")
    print(f"  F1-Score    : {f1:.4f}")
    print(f"  AUC-ROC     : {auc:.4f}")
    print()
    print(classification_report(y_test, y_pred, target_names=["Negativo", "Positivo"]))

    return y_pred, y_prob, cm


# Evaluamos ambos modelos
y_pred_base, y_prob_base, cm_base = evaluate_model(model_base, X_test_s, y_test, "Modelo BASE")
y_pred_reg,  y_prob_reg,  cm_reg  = evaluate_model(model_reg,  X_test_s, y_test, "Modelo REGULARIZADO")

## 8. Gráfica 2 – Matrices de Confusión (Base vs Regularizado)

La matriz de confusión permite visualizar **dónde falla cada modelo**: falsos positivos (FP) y falsos negativos (FN), que tienen implicaciones clínicas distintas en el diagnóstico de diabetes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Matriz de confusión Modelo BASE ─────────────────────────────────────────
disp_base = ConfusionMatrixDisplay(confusion_matrix=cm_base,
                                   display_labels=["Negativo", "Positivo"])
disp_base.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Modelo BASE\n(sin regularización)", fontsize=12, fontweight="bold")

# ── Matriz de confusión Modelo REGULARIZADO ──────────────────────────────────
disp_reg = ConfusionMatrixDisplay(confusion_matrix=cm_reg,
                                  display_labels=["Negativo", "Positivo"])
disp_reg.plot(ax=axes[1], colorbar=False, cmap="Greens")
axes[1].set_title("Modelo REGULARIZADO\n(L2 + Dropout + Early Stopping)", fontsize=12, fontweight="bold")

plt.suptitle("Gráfica 2 – Matrices de Confusión: Comparación de Predicciones",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfica guardada como confusion_matrices.png")

## 9. Tabla resumen comparativa

In [ ]:
# Recalculamos métricas de forma compacta para la tabla comparativa
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

metrics_summary = {
    "Modelo"          : ["BASE (sin reg.)", "REGULARIZADO (L2+Dropout+ES)"],
    "Test Loss"       : [round(loss_base, 4), round(loss_reg, 4)],
    "Test Accuracy"   : [round(acc_base, 4),  round(acc_reg, 4)],
    "F1-Score"        : [round(f1_score(y_test, y_pred_base), 4),
                         round(f1_score(y_test, y_pred_reg),  4)],
    "AUC-ROC"         : [round(roc_auc_score(y_test, y_prob_base), 4),
                         round(roc_auc_score(y_test, y_prob_reg),  4)],
    "Épocas"          : [len(history_base.history["loss"]),
                         len(history_reg.history["loss"])],
}

# Imprimimos la tabla
col_w = [30, 12, 15, 10, 10, 8]
headers = list(metrics_summary.keys())
print(" | ".join(h.ljust(col_w[i]) for i, h in enumerate(headers)))
print("-" * 95)
for row in range(2):
    vals = [str(metrics_summary[h][row]) for h in headers]
    print(" | ".join(v.ljust(col_w[i]) for i, v in enumerate(vals)))

## 10. Conclusiones

1. **Overfitting en el modelo base:** La curva de `train_loss` del modelo sin regularización desciende sostenidamente, mientras que `val_loss` comienza a subir o estancarse, evidenciando una brecha train/val característica del sobreajuste (alta varianza).

2. **Efecto del Dropout:** Al desactivar aleatoriamente el 30 % de neuronas en cada paso de entrenamiento, el modelo no puede memorizar los datos; esto fuerza representaciones más robustas y reduce la brecha entre `train_loss` y `val_loss`.

3. **Efecto de la regularización L2:** El término de penalización `λ||w||²` desincentiva pesos muy grandes, lo que equivale a un prior bayesiano que empuja los parámetros hacia cero y mejora la generalización.

4. **Early Stopping como barrera al overfitting:** Detener el entrenamiento en el epoch de mínima `val_loss` y restaurar esos pesos garantiza que usemos la versión del modelo con mejor capacidad de generalización, sin necesidad de fijar el número de épocas manualmente.

5. **Trade-off bias/variance observado:** El modelo regularizado sacrifica un pequeño margen de accuracy en train (incremento de bias) a cambio de mejor desempeño y estabilidad en test (reducción de varianza), lo que se traduce en métricas como F1-Score y AUC-ROC iguales o superiores a las del modelo base en datos no vistos.